# Experiment 002 — Full Pipeline with Extended Audio Features

**Replicates the entire exp001 pipeline** using an extended feature set:
the **full FRESH-selected feature set** from exp001 (~326 features) **plus**
additional handcrafted audio features (tempo, spectral rolloff/flatness, onset
strength, fractal dimensions, TKEO, ISI statistics).

All outputs are written to **`results_ext/`** (instead of `results/`) so that
exp001 artefacts remain untouched.

## Pipeline Overview

1. **Extended Feature Extraction** — Tempo, spectral, onset, fractal, TKEO, ISI
2. **Feature Merging** — Combine full FRESH feature set + extended features
3. **Grid Search** — 5-fold CV for RF, XGB, SVM, MLP, kNN
4. **Statistical Analysis** — Friedman test + Nemenyi post-hoc + CD diagram
5. **RFE Feature Elimination** — Recursive elimination with CV
6. **XGBoost Fine-tuning** — Optimization on reduced feature set (1-SE rule)
7. **Feature Importance** — Impurity, Permutation, SHAP analysis
8. **Final Evaluation** — Test set metrics and confusion matrix
9. **Export** — PDF figures, LaTeX tables, C code for deployment
10. **Soft-Voting Ensemble** — XGB + RF + SVM (1-to-1 comparison with exp001)
11. **Conformal Prediction** — Cross-conformal with LAC & APS
12. **Export** — LaTeX tables for ensemble & conformal (+ cross-experiment comparison)
13. **Exploration** — k=17 RFE features + extended features (targeted integration)


In [ ]:
# ── Setup: path, project modules, all imports ─────────────────────────────────
import sys, json, shutil
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Project modules
from src import config, preprocessing, features, modeling, visualization

# Scientific / data
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# Audio / signal processing
import librosa
import antropy
import scipy.signal
from scipy.stats import skew, kurtosis, entropy as scipy_entropy
from tqdm import tqdm

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    balanced_accuracy_score, precision_score, recall_score,
)
from sklearn.inspection import permutation_importance

# XGBoost
from xgboost import XGBClassifier

# SHAP
import shap

# Autorank
import autorank

# Conformal Prediction
from mapie.classification import CrossConformalClassifier

# ── results_ext directory tree ────────────────────────────────────────────────
RESULTS_EXT_DIR      = config.PROJECT_ROOT / "results_ext"
GRID_SEARCH_EXT      = RESULTS_EXT_DIR / "grid_search"
RFE_EXT              = RESULTS_EXT_DIR / "rfe"
FINETUNE_EXT         = RESULTS_EXT_DIR / "finetuned_models"
IMPORTANCE_EXT       = RESULTS_EXT_DIR / "feature_importance"
METRICS_EXT          = RESULTS_EXT_DIR / "metrics"
LATEX_FIGURES_EXT    = RESULTS_EXT_DIR / "latex_figures"
LATEX_TABLES_EXT     = RESULTS_EXT_DIR / "latex_tables"
TL2CGEN_EXT          = RESULTS_EXT_DIR / "tl2cgen_export"
ENSEMBLE_EXT         = RESULTS_EXT_DIR / "ensemble"
CONFORMAL_EXT        = RESULTS_EXT_DIR / "conformal"

for d in [RESULTS_EXT_DIR, GRID_SEARCH_EXT, RFE_EXT, FINETUNE_EXT,
          IMPORTANCE_EXT, METRICS_EXT, LATEX_FIGURES_EXT, LATEX_TABLES_EXT,
          TL2CGEN_EXT, ENSEMBLE_EXT, CONFORMAL_EXT]:
    d.mkdir(parents=True, exist_ok=True)

# Create original output directories too (needed for loading exp001 artefacts)
config.create_directories()

# Paths for extended feature CSVs
EXTENDED_FEATURES_CSV = config.INTERIM_DIR / "extended_audio_features.csv"
EXTENDED_TRAIN_CSV    = config.PROCESSED_DIR / "exp02_extended_train.csv"
EXTENDED_TEST_CSV     = config.PROCESSED_DIR / "exp02_extended_test.csv"
EXTENDED_FEATURES_TXT = config.PROCESSED_DIR / "exp02_extended_features.txt"

SR = config.SAMPLE_RATE  # 800 Hz

print(f"Project root  : {config.PROJECT_ROOT}")
print(f"Results (ext) : {RESULTS_EXT_DIR}")
print(f"\n✓ All modules loaded, results_ext/ directories created")

In [ ]:
# Fix absolute paths in consolidated_summary.csv to current project directory
orig = config.CONSOLIDATED_SUMMARY
backup = config.CONSOLIDATED_SUMMARY.with_suffix('.csv.bak')

if not orig.exists():
    print(f"Consolidated summary not found: {orig}")
else:
    if not backup.exists():
        shutil.copy2(orig, backup)
        print(f"Backed up original consolidated summary to: {backup}")

    df = pd.read_csv(orig)
    df['path'] = df['filename'].apply(lambda fn: str(config.CONSOLIDATED_DIR / fn))
    df.to_csv(orig, index=False)
    print(f"Rewrote paths in: {orig}")
    print(df.head())

## 1. Extended Audio Feature Extraction

Extract additional handcrafted audio features: tempo, spectral rolloff/flatness,
onset strength, fractal dimensions (Higuchi, Petrosian), TKEO, and
Inter-Step Interval (ISI) statistics.

> Set **`FORCE_REEXTRACT = True`** to re-extract even if the CSV cache exists.

In [ ]:
FORCE_REEXTRACT = False


def compute_tkeo(x: np.ndarray) -> np.ndarray:
    """Teager-Kaiser Energy Operator."""
    if len(x) < 3:
        return np.array([0.0])
    return x[1:-1] ** 2 - x[:-2] * x[2:]


def compute_isi_features(y: np.ndarray, sr: int, min_distance_sec: float = 0.15) -> dict:
    """Inter-Step Interval (ISI) features derived from onset-envelope peaks."""
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    hop        = 512
    min_dist   = max(1, int(min_distance_sec * sr / hop))
    threshold  = float(np.percentile(onset_env, 50))

    peaks, _ = scipy.signal.find_peaks(onset_env, height=threshold, distance=min_dist)
    duration_sec = len(y) / sr

    if len(peaks) < 2:
        return dict(
            isi_mean=0.0, isi_std=0.0, isi_min=0.0, isi_max=0.0,
            isi_range=0.0, isi_median=0.0, isi_iqr=0.0,
            isi_skewness=0.0, isi_kurtosis=0.0,
            isi_cv=0.0, isi_rmssd=0.0, isi_acf1=0.0, isi_entropy=0.0,
            n_steps=int(len(peaks)),
            steps_per_sec=float(len(peaks)) / max(duration_sec, 1e-6),
        )

    peak_times = librosa.frames_to_time(peaks, sr=sr, hop_length=hop)
    isi = np.diff(peak_times)
    n   = len(isi)

    isi_mean  = float(np.mean(isi))
    isi_std   = float(np.std(isi))
    isi_min   = float(np.min(isi))
    isi_max   = float(np.max(isi))
    isi_range = isi_max - isi_min

    isi_median = float(np.median(isi))
    q75, q25   = np.percentile(isi, [75, 25])
    isi_iqr    = float(q75 - q25)
    isi_skewness = float(skew(isi))      if n >= 3 else 0.0
    isi_kurtosis = float(kurtosis(isi))  if n >= 4 else 0.0

    isi_cv    = isi_std / (isi_mean + 1e-10)
    isi_rmssd = float(np.sqrt(np.mean(np.diff(isi) ** 2))) if n >= 2 else 0.0

    if n >= 3:
        isi_z    = isi - isi_mean
        isi_acf1 = float(np.dot(isi_z[:-1], isi_z[1:]) /
                         (np.dot(isi_z, isi_z) + 1e-10))
    else:
        isi_acf1 = 0.0

    hist, _ = np.histogram(isi, bins=min(8, n), density=False)
    hist = hist + 1e-10
    hist = hist / hist.sum()
    isi_entropy = float(scipy_entropy(hist))

    return dict(
        isi_mean=isi_mean, isi_std=isi_std,
        isi_min=isi_min,   isi_max=isi_max, isi_range=isi_range,
        isi_median=isi_median, isi_iqr=isi_iqr,
        isi_skewness=isi_skewness, isi_kurtosis=isi_kurtosis,
        isi_cv=isi_cv, isi_rmssd=isi_rmssd,
        isi_acf1=isi_acf1, isi_entropy=isi_entropy,
        n_steps=int(len(peaks)),
        steps_per_sec=float(len(peaks)) / max(duration_sec, 1e-6),
    )


def extract_extended_features(audio_path, sr: int = SR) -> dict:
    """Extract all additional audio features for a single file."""  # noqa
    try:
        y   = preprocessing.load_and_preprocess_audio(Path(audio_path))
        y64 = y.astype(np.float64)
        feat = {}

        # 1. Tempo / Cadence
        tempo_result, _ = librosa.beat.beat_track(y=y64, sr=sr)
        feat["tempo"] = float(np.atleast_1d(tempo_result)[0])

        # 2. Spectral Rolloff
        rolloff = librosa.feature.spectral_rolloff(y=y64, sr=sr, roll_percent=0.85)
        feat["spectral_rolloff_mean"] = float(np.mean(rolloff))
        feat["spectral_rolloff_std"]  = float(np.std(rolloff))

        # 3. Spectral Flatness
        flatness = librosa.feature.spectral_flatness(y=y64)
        feat["spectral_flatness_mean"] = float(np.mean(flatness))
        feat["spectral_flatness_std"]  = float(np.std(flatness))

        # 4. Onset Strength
        onset_env = librosa.onset.onset_strength(y=y64, sr=sr)
        feat["onset_strength_mean"] = float(np.mean(onset_env))
        feat["onset_strength_std"]  = float(np.std(onset_env))
        feat["onset_strength_max"]  = float(np.max(onset_env))
        feat["onset_strength_sum"]  = float(np.sum(onset_env))

        # 5. Higuchi Fractal Dimension
        try:
            feat["higuchi_fd"] = float(antropy.higuchi_fd(y64, kmax=10))
        except Exception:
            feat["higuchi_fd"] = 0.0

        # 6. Petrosian Fractal Dimension
        try:
            feat["petrosian_fd"] = float(antropy.petrosian_fd(y64))
        except Exception:
            feat["petrosian_fd"] = 0.0

        # 7. TKEO
        tkeo = compute_tkeo(y64)
        feat["tkeo_mean"] = float(np.mean(tkeo))
        feat["tkeo_std"]  = float(np.std(tkeo))
        feat["tkeo_max"]  = float(np.max(np.abs(tkeo)))

        # 8. ISI features
        feat.update(compute_isi_features(y64, sr))

        return feat

    except Exception as e:
        print(f"  ✗ Error processing {audio_path}: {e}")
        return {}


# ── Load consolidated summary ─────────────────────────────────────────────────
consolidated_df = pd.read_csv(config.CONSOLIDATED_SUMMARY)
print(f"Total samples: {len(consolidated_df)}")

# ── Extract (or reload if cached) ────────────────────────────────────────────
if EXTENDED_FEATURES_CSV.exists() and not FORCE_REEXTRACT:
    print(f"✓ Loading existing extended features:\n  {EXTENDED_FEATURES_CSV}")
    extended_df = pd.read_csv(EXTENDED_FEATURES_CSV)
else:
    if FORCE_REEXTRACT:
        print("FORCE_REEXTRACT=True — re-extracting features…")
    else:
        print(f"Extracting extended features for {len(consolidated_df)} samples…")

    all_records = []
    for _, row in tqdm(consolidated_df.iterrows(), total=len(consolidated_df), desc="Extracting"):
        feat = extract_extended_features(row["path"])
        if feat:
            feat["filename"] = row["filename"]
            feat["category"] = row["category"]
            feat["target"]   = row["target"]
            all_records.append(feat)

    extended_df = pd.DataFrame(all_records)
    extended_df.to_csv(EXTENDED_FEATURES_CSV, index=False)
    print(f"✓ Saved to: {EXTENDED_FEATURES_CSV}")

print(f"\nExtended features shape: {extended_df.shape}")
ext_feature_cols = [c for c in extended_df.columns if c not in ["filename", "category", "target"]]
print(f"Feature columns ({len(ext_feature_cols)}): {ext_feature_cols}")

## 2. Merge Full FRESH Feature Set + Extended Audio Features

Load **all FRESH-selected features** from exp001 (`config.FRESH_FEATURES`, ~326 features),
align the extended features to the same train/test split, and create combined DataFrames.


In [ ]:
# ── 1. Load all FRESH-selected feature names ──────────────────────────────────
with open(config.FRESH_FEATURES) as fh:
    fresh_features = [l.strip() for l in fh if l.strip()]
print(f"FRESH-selected features loaded: {len(fresh_features)}")

# ── 2. Load the existing FRESH train / test splits ───────────────────────────
train_df = pd.read_csv(config.FRESH_TRAIN)
test_df  = pd.read_csv(config.FRESH_TEST)

# ── 3. Recover sample order via the same deterministic split ─────────────────
dataset_clean   = pd.read_csv(config.DATASET_CLEAN)
consolidated_df = pd.read_csv(config.CONSOLIDATED_SUMMARY)

dataset_clean = dataset_clean.copy()
dataset_clean['filename'] = consolidated_df['filename'].values[:len(dataset_clean)]

_y_all = dataset_clean['target']
_idx   = np.arange(len(dataset_clean))
_idx_train, _idx_test = train_test_split(
    _idx, test_size=config.TEST_SIZE, stratify=_y_all,
    random_state=config.RANDOM_STATE
)

train_filenames = dataset_clean.iloc[_idx_train]['filename'].str.replace('.wav', '').values
test_filenames  = dataset_clean.iloc[_idx_test]['filename'].str.replace('.wav', '').values

# ── 4. Build extended-feature lookup ─────────────────────────────────────────
ext_feature_cols = [c for c in extended_df.columns
                    if c not in ['filename', 'category', 'target']]

extended_lookup = (
    extended_df
    .assign(stem=extended_df['filename'].str.replace('.wav', ''))
    .set_index('stem')[ext_feature_cols]
)
print(f"Extended feature columns ({len(ext_feature_cols)}): {ext_feature_cols}")

# ── 5. Align extended features to train and test rows ────────────────────────
ext_train = extended_lookup.reindex(train_filenames).fillna(0).reset_index(drop=True)
ext_test  = extended_lookup.reindex(test_filenames).fillna(0).reset_index(drop=True)

# ── 6. Build combined DataFrames ─────────────────────────────────────────────
combined_features = fresh_features + ext_feature_cols

train_ext_df = pd.concat(
    [train_df[['category'] + fresh_features + ['target']].reset_index(drop=True),
     ext_train],
    axis=1,
)
test_ext_df = pd.concat(
    [test_df[['category'] + fresh_features + ['target']].reset_index(drop=True),
     ext_test],
    axis=1,
)

print(f"\nCombined feature set: FRESH ({len(fresh_features)}) + extended ({len(ext_feature_cols)}) = {len(combined_features)} features")
print(f"Train: {len(train_ext_df)} samples  |  Test: {len(test_ext_df)} samples")

# ── 7. Save combined CSV and feature list ────────────────────────────────────
train_ext_df.to_csv(EXTENDED_TRAIN_CSV, index=False)
test_ext_df.to_csv(EXTENDED_TEST_CSV, index=False)

with open(EXTENDED_FEATURES_TXT, 'w') as fh:
    fh.write('\n'.join(combined_features))

print(f"\n✓ Saved combined train ({len(train_ext_df)} rows): {EXTENDED_TRAIN_CSV}")
print(f"✓ Saved combined test  ({len(test_ext_df)} rows) : {EXTENDED_TEST_CSV}")
print(f"✓ Feature list ({len(combined_features)} features): {EXTENDED_FEATURES_TXT}")


## 3. Grid Search — Model Comparison

5-fold CV grid search for RF, XGB, SVM, MLP, kNN on the combined (k=17 + extended) feature set.
Results are saved to `results_ext/grid_search/`.

In [ ]:
# Load combined data
train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)

with open(EXTENDED_FEATURES_TXT) as fh:
    combined_features = [l.strip() for l in fh if l.strip()]

X_train = train_ext_df[combined_features].values
y_train = train_ext_df['target'].values
X_test  = test_ext_df[combined_features].values
y_test  = test_ext_df['target'].values

print(f"Train: {X_train.shape}")
print(f"Test : {X_test.shape}\n")

# Run incremental grid search → results_ext/grid_search/
summary_df = modeling.run_incremental_grid_search(
    X_train, y_train, X_test, y_test,
    output_dir=GRID_SEARCH_EXT,
)

print("\n" + "=" * 60)
print("Summary (sorted by CV score):")
print(summary_df[['model', 'best_cv_score', 'cv_std', 'test_score', 'n_combinations']].to_string(index=False))

### 3b. Grid Search — XGBoost only

`run_incremental_grid_search` skips XGB by design
(`model_names = ["RF", "SVM", "MLP", "kNN"]`).
Run it separately here using `run_grid_search` with the standard `config.PARAM_GRIDS["XGB"]`,
then merge the result back into `summary_df`.


In [ ]:
# Re-load data (safe to re-run; no change to other model results)
train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)

with open(EXTENDED_FEATURES_TXT) as fh:
    combined_features = [l.strip() for l in fh if l.strip()]

X_train = train_ext_df[combined_features].values
y_train = train_ext_df['target'].values
X_test  = test_ext_df[combined_features].values
y_test  = test_ext_df['target'].values

# Run XGB grid search only — saves XGB_fold_statistics.json + XGB_cv_results.csv
# to GRID_SEARCH_EXT, exactly as run_grid_search would for any other model.
xgb_summary = modeling.run_grid_search(
    X_train, y_train, X_test, y_test,
    output_dir=GRID_SEARCH_EXT,
    model_names=["XGB"],
)
print(xgb_summary[['model', 'best_cv_score', 'cv_std', 'test_score', 'n_combinations']].to_string(index=False))

# ── Merge XGB row into the existing summary and re-save ───────────────────────
existing_summary_path = GRID_SEARCH_EXT / "summary.csv"
if existing_summary_path.exists():Ae
    existing_df = pd.read_csv(existing_summary_path)
    # Drop stale XGB row if present, then append the fresh one
    existing_df = existing_df[existing_df['model'] != 'XGB']
    summary_df = (
        pd.concat([existing_df, xgb_summary], ignore_index=True)
        .sort_values('best_cv_score', ascending=False)
        .reset_index(drop=True)
    )
else:
    summary_df = xgb_summary

summary_df.to_csv(existing_summary_path, index=False)

print("\nUpdated summary (all models):")
print(summary_df[['model', 'best_cv_score', 'cv_std', 'test_score', 'n_combinations']].to_string(index=False))
print(f"\n✓ Updated summary saved → {existing_summary_path}")


## 4. Statistical Model Comparison

Friedman test and Nemenyi post-hoc analysis on per-fold CV scores.

In [ ]:
# Load fold scores from results_ext/grid_search/
fold_scores = modeling.load_fold_scores(results_dir=GRID_SEARCH_EXT)

print("CV Fold Scores:")
for model, scores in fold_scores.items():
    print(f"  {model}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# Run statistical comparison → results_ext/grid_search/
stats_results = modeling.run_statistical_comparison(fold_scores, output_dir=GRID_SEARCH_EXT)

## 5. Critical Difference Diagram

In [ ]:
# Generate CD diagram with autorank
score_df = stats_results['fold_scores']

result = autorank.autorank(score_df, alpha=0.05, verbose=False, order='descending')

fig, ax = plt.subplots(figsize=(10, 2.5))
autorank.plot_stats(result, ax=ax, allow_insignificant=True)
plt.title('Confidence Interval Plot — Extended Features (α = 0.05)', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()

# Save → results_ext/latex_figures/
cd_path = LATEX_FIGURES_EXT / "confidence_interval_plot.pdf"
plt.savefig(cd_path, format='pdf', bbox_inches='tight', dpi=300)
plt.show()

print(f"✓ Saved: {cd_path}")
print("\n" + "=" * 60)
print(result)

## 6. RFE Feature Elimination

Recursive Feature Elimination on the combined feature set to find the optimal
feature subset. XGBoost best params are loaded from `results_ext/grid_search/`.

In [ ]:
# Load combined data
train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)

with open(EXTENDED_FEATURES_TXT) as fh:
    combined_features = [l.strip() for l in fh if l.strip()]

X_train = train_ext_df[combined_features]
y_train = train_ext_df['target'].values
feature_names = list(X_train.columns)

print(f"Features: {len(feature_names)}")

# Load XGB params from the extended grid search
with open(GRID_SEARCH_EXT / "XGB_fold_statistics.json", 'r') as f:
    xgb_stats = json.load(f)
xgb_params = {k.replace('clf__', ''): v for k, v in xgb_stats['best_params'].items()
               if k.startswith('clf__')}

print(f"XGB params: {xgb_params}")

# Run RFE → results_ext/rfe/
ranking_df = modeling.run_rfe_elimination(
    X_train.values, y_train, feature_names,
    xgb_params=xgb_params, output_dir=RFE_EXT,
)
ordered_features = ranking_df['feature'].tolist()

# Evaluate k-path
k_path_df, k_summary = modeling.evaluate_k_path(
    X_train, y_train, ordered_features,
    xgb_params=xgb_params, output_dir=RFE_EXT,
)

# Select optimal k
k_selection = modeling.select_optimal_k(k_summary, output_dir=RFE_EXT)

# Save feature lists
for method, k in [('best', k_selection['best_k']), ('1se', k_selection['k_1se']),
                  ('friedman', k_selection['k_friedman'])]:
    features_k = ordered_features[:k]
    with open(RFE_EXT / f"rfe_features_{method}_k{k}.txt", 'w') as f:
        f.write('\n'.join(features_k))

print(f"\n✓ Feature lists saved to: {RFE_EXT}")

## 7. XGBoost Fine-tuning

Optimize XGBoost on the reduced feature set selected by the 1-SE rule.

In [ ]:
# Select k based on 1-SE rule
SELECTED_K = k_selection['k_1se']
selected_features = ordered_features[:SELECTED_K]

print(f"Selected k: {SELECTED_K} features")

# Prepare data
train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)

X_train_k = train_ext_df[selected_features].values
y_train   = train_ext_df['target'].values
X_test_k  = test_ext_df[selected_features].values
y_test    = test_ext_df['target'].values

print(f"Training shape: {X_train_k.shape}")
print(f"Test shape    : {X_test_k.shape}\n")

# Fine-tune → results_ext/finetuned_models/
finetuned_model, finetune_results = modeling.finetune_xgb(
    X_train_k, y_train, X_test_k, y_test, SELECTED_K,
    output_dir=FINETUNE_EXT,
)

# Save selected features
with open(FINETUNE_EXT / f"selected_features_k{SELECTED_K}.txt", 'w') as f:
    f.write('\n'.join(selected_features))

print(f"\n✓ Fine-tuned model + selected features saved to: {FINETUNE_EXT}")

## 8. Statistical Comparison: Original vs Fine-tuned

Wilcoxon signed-rank test between the full-feature XGBoost from the grid search
and the fine-tuned XGBoost on the reduced feature set.

In [ ]:
# Load original XGBoost fold scores from extended grid search
with open(GRID_SEARCH_EXT / "XGB_fold_statistics.json", 'r') as f:
    original_xgb = json.load(f)

original_fold_scores = [original_xgb[f'split{i}_test_score'] for i in range(5)]
finetune_fold_scores = finetune_results['fold_scores']

# Comparison
wilcoxon_result = modeling.compare_models_wilcoxon(
    original_fold_scores, finetune_fold_scores,
    f"Original XGB ({len(combined_features)} features)",
    f"Fine-tuned XGB ({SELECTED_K} features)",
)

# Save comparison
comparison_results = {
    'original_xgb': {
        'mean_cv_score': np.mean(original_fold_scores),
        'std_cv_score': np.std(original_fold_scores),
        'n_features': len(combined_features),
        'n_estimators': original_xgb['best_params'].get('clf__n_estimators', 1000),
    },
    'finetuned_xgb': {
        'mean_cv_score': np.mean(finetune_fold_scores),
        'std_cv_score': np.std(finetune_fold_scores),
        'n_features': SELECTED_K,
        'n_estimators': finetune_results['best_params'].get('clf__n_estimators', 100),
    },
    'wilcoxon_test': wilcoxon_result,
}

with open(FINETUNE_EXT / "comparison_results.json", 'w') as f:
    json.dump(comparison_results, f, indent=2)

print(f"\n✓ Comparison saved to: {FINETUNE_EXT / 'comparison_results.json'}")

## 9. Feature Importance Analysis

Impurity-based, Permutation, and SHAP importance for the fine-tuned XGBoost model.

In [ ]:
# Load model and data
finetuned_model = joblib.load(FINETUNE_EXT / f"xgb_finetuned_k{SELECTED_K}.pkl")

train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)

X_train_k = train_ext_df[selected_features].values
X_test_k  = test_ext_df[selected_features].values
y_test    = test_ext_df['target'].values

# Get class names
class_mapping = train_ext_df[['category', 'target']].drop_duplicates().set_index('target')['category'].to_dict()
class_names = [class_mapping[i] for i in sorted(class_mapping.keys())]

# Humanize feature names
humanized_features = features.humanize_feature_names(selected_features)

print(f"Model: Fine-tuned XGBoost with {SELECTED_K} features")
print(f"Classes: {class_names}")

# Compute importance analysis → results_ext/feature_importance/
importance_results = modeling.compute_importance_analysis(
    finetuned_model, X_train_k, X_test_k, y_test,
    selected_features, humanized_features, class_names,
    output_dir=IMPORTANCE_EXT,
)

print(f"\n✓ All importance results saved to: {IMPORTANCE_EXT}")

## 10. Final Evaluation on Test Set

In [ ]:
# Final evaluation → results_ext/metrics/
final_results = modeling.evaluate_final_model(
    finetuned_model, X_test_k, y_test, class_names,
    output_dir=METRICS_EXT,
)

print(f"\n✓ Final metrics saved to: {METRICS_EXT}")

# 10a. McNemar test

In [ ]:
import json
from statsmodels.stats.contingency_tables import mcnemar

METRICS_EXT.mkdir(parents=True, exist_ok=True)

# ── Paths ─────────────────────────────────────────────────────────────────────
exp001_test_csv = config.FRESH_TEST
exp002_test_csv = EXTENDED_TEST_CSV if 'EXTENDED_TEST_CSV' in globals() else (config.PROCESSED_DIR / 'exp02_extended_test.csv')

exp001_features_path = config.FINETUNE_DIR / 'selected_features_k17.txt'
exp002_features_path = FINETUNE_EXT / f'selected_features_k{SELECTED_K}.txt'

exp001_model_path = config.FINETUNE_DIR / 'xgb_finetuned_k17.pkl'
exp002_model_path = FINETUNE_EXT / f'xgb_finetuned_k{SELECTED_K}.pkl'

comparison_csv = METRICS_EXT / 'final_model_comparison_exp001_vs_exp002.csv'
predictions_csv = METRICS_EXT / 'final_model_predictions_exp001_vs_exp002.csv'
contingency_csv = METRICS_EXT / 'mcnemar_contingency_exp001_vs_exp002.csv'
mcnemar_json = METRICS_EXT / 'mcnemar_exp001_vs_exp002.json'

# ── Load aligned test splits ─────────────────────────────────────────────────
exp001_test_df = pd.read_csv(exp001_test_csv)
exp002_test_df = pd.read_csv(exp002_test_csv)

if len(exp001_test_df) != len(exp002_test_df):
    raise ValueError(
        f'Mismatched test sizes: exp001={len(exp001_test_df)} vs exp002={len(exp002_test_df)}'
    )

if not exp001_test_df['target'].equals(exp002_test_df['target']):
    raise ValueError('exp001 and exp002 test splits are not aligned row-by-row.')

# ── Load selected features and saved final models ────────────────────────────
with open(exp001_features_path, 'r') as f:
    exp001_features = [line.strip() for line in f if line.strip()]

with open(exp002_features_path, 'r') as f:
    exp002_features = [line.strip() for line in f if line.strip()]

exp001_model = joblib.load(exp001_model_path)
exp002_model = joblib.load(exp002_model_path)

X_test_exp001 = exp001_test_df[exp001_features].values
X_test_exp002 = exp002_test_df[exp002_features].values
y_true = exp001_test_df['target'].values

y_pred_exp001 = exp001_model.predict(X_test_exp001)
y_pred_exp002 = exp002_model.predict(X_test_exp002)

correct_exp001 = y_pred_exp001 == y_true
correct_exp002 = y_pred_exp002 == y_true

# ── Build metric comparison table from saved final metrics ───────────────────
exp001_metrics = pd.read_csv(config.METRICS_DIR / 'final_model_metrics.csv').set_index('Metric')['Value']
exp002_metrics = pd.read_csv(METRICS_EXT / 'final_model_metrics.csv').set_index('Metric')['Value']

metric_keys = [
    'Accuracy', 'Balanced Accuracy', 'F1 (Macro)',
    'F1 (Weighted)', 'Precision (Macro)', 'Recall (Macro)'
]

comparison_rows = []
for metric_name in metric_keys:
    value_exp001 = float(exp001_metrics[metric_name])
    value_exp002 = float(exp002_metrics[metric_name])
    comparison_rows.append({
        'Metric': metric_name,
        'exp001 (k=17)': round(value_exp001, 4),
        f'exp002 (k={SELECTED_K}, ext)': round(value_exp002, 4),
        'Δ (exp002 − exp001)': round(value_exp002 - value_exp001, 4),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Metric')
comparison_df.to_csv(comparison_csv)

# ── Save per-sample prediction table ─────────────────────────────────────────
predictions_df = pd.DataFrame({
    'test_row': np.arange(len(y_true)),
    'category': exp001_test_df['category'].values,
    'target': y_true,
    'exp001_pred': y_pred_exp001,
    'exp002_pred': y_pred_exp002,
    'exp001_correct': correct_exp001.astype(int),
    'exp002_correct': correct_exp002.astype(int),
})
predictions_df.to_csv(predictions_csv, index=False)

# ── McNemar contingency table ────────────────────────────────────────────────
a = int((correct_exp002 & correct_exp001).sum())
b = int((correct_exp002 & ~correct_exp001).sum())
c = int((~correct_exp002 & correct_exp001).sum())
d = int((~correct_exp002 & ~correct_exp001).sum())

contingency_df = pd.DataFrame(
    [[a, b], [c, d]],
    index=['exp002 correct', 'exp002 wrong'],
    columns=['exp001 correct', 'exp001 wrong'],
)
contingency_df.to_csv(contingency_csv)

exact_res = mcnemar(contingency_df.to_numpy(), exact=True)
chi2_res = mcnemar(contingency_df.to_numpy(), exact=False, correction=True)

results_payload = {
    'n_test_samples': int(len(y_true)),
    'exp001_model': 'xgb_finetuned_k17',
    'exp002_model': f'xgb_finetuned_k{SELECTED_K}',
    'exp001_accuracy': float(correct_exp001.mean()),
    'exp002_accuracy': float(correct_exp002.mean()),
    'accuracy_delta_exp002_minus_exp001': float(correct_exp002.mean() - correct_exp001.mean()),
    'b_exp002_correct_exp001_wrong': b,
    'c_exp002_wrong_exp001_correct': c,
    'mcnemar_exact_pvalue': float(exact_res.pvalue),
    'mcnemar_chi2_cc_stat': float(chi2_res.statistic),
    'mcnemar_chi2_cc_pvalue': float(chi2_res.pvalue),
}

with open(mcnemar_json, 'w') as f:
    json.dump(results_payload, f, indent=2)

print('Final test-set comparison: exp001 vs exp002\n')
print(comparison_df.to_string(float_format='%.4f'))
print('\nMcNemar contingency table:')
print(contingency_df.to_string())
print(f"\nDiscordant counts (off-diagonal): b={b}, c={c}")
print('\nMcNemar test (statsmodels):')
print(f"  Exact binomial p-value           : {exact_res.pvalue:.6f}")
print(f"  Chi-square + continuity corr.    : stat={chi2_res.statistic:.4f}, p={chi2_res.pvalue:.6f}")

alpha = 0.05
if exact_res.pvalue < alpha:
    print('\nResult: statistically significant difference at alpha = 0.05.')
else:
    print('\nResult: no statistically significant difference at alpha = 0.05.')

print(f"\n✓ Saved metric comparison   → {comparison_csv}")
print(f"✓ Saved prediction table    → {predictions_csv}")
print(f"✓ Saved contingency table   → {contingency_csv}")
print(f"✓ Saved McNemar summary     → {mcnemar_json}")


# 10b. Paired bootstrap

In [ ]:
# ── Stratified paired bootstrap: exp001 vs exp002 final models ───────────────
from sklearn.utils import resample

N_BOOTSTRAP = 10_000
BOOTSTRAP_CI = 0.95
BOOTSTRAP_RANDOM_STATE = config.RANDOM_STATE

bootstrap_samples_csv = METRICS_EXT / 'bootstrap_final_model_metrics_exp001_vs_exp002.csv'
bootstrap_summary_csv = METRICS_EXT / 'bootstrap_final_model_summary_exp001_vs_exp002.csv'
bootstrap_json = METRICS_EXT / 'bootstrap_final_model_exp001_vs_exp002.json'

# Reuse arrays from 10b when available; otherwise load the saved prediction table.
if not all(name in globals() for name in ['y_true', 'y_pred_exp001', 'y_pred_exp002']):
    predictions_df = pd.read_csv(METRICS_EXT / 'final_model_predictions_exp001_vs_exp002.csv')
    y_true = predictions_df['target'].values
    y_pred_exp001 = predictions_df['exp001_pred'].values
    y_pred_exp002 = predictions_df['exp002_pred'].values

y_true_boot = np.asarray(y_true)
y_pred_exp001_boot = np.asarray(y_pred_exp001)
y_pred_exp002_boot = np.asarray(y_pred_exp002)
bootstrap_labels = np.sort(np.unique(y_true_boot))

def _bootstrap_metric_dict(y_true_sample, y_pred_sample, labels):
    return {
        'Accuracy': accuracy_score(y_true_sample, y_pred_sample),
        'Balanced Accuracy': balanced_accuracy_score(y_true_sample, y_pred_sample),
        'Precision (Macro)': precision_score(
            y_true_sample, y_pred_sample, labels=labels,
            average='macro', zero_division=0,
        ),
        'Recall (Macro)': recall_score(
            y_true_sample, y_pred_sample, labels=labels,
            average='macro', zero_division=0,
        ),
        'F1 (Macro)': f1_score(
            y_true_sample, y_pred_sample, labels=labels,
            average='macro', zero_division=0,
        ),
        'F1 (Weighted)': f1_score(
            y_true_sample, y_pred_sample, labels=labels,
            average='weighted', zero_division=0,
        ),
    }

observed_exp001 = _bootstrap_metric_dict(y_true_boot, y_pred_exp001_boot, bootstrap_labels)
observed_exp002 = _bootstrap_metric_dict(y_true_boot, y_pred_exp002_boot, bootstrap_labels)
metric_names = list(observed_exp001.keys())

bootstrap_rows = []

for replicate in range(N_BOOTSTRAP):
    sample_true, sample_pred_exp001, sample_pred_exp002 = resample(
        y_true_boot,
        y_pred_exp001_boot,
        y_pred_exp002_boot,
        replace=True,
        n_samples=len(y_true_boot),
        stratify=y_true_boot,
        random_state=BOOTSTRAP_RANDOM_STATE + replicate,
    )

    sample_metrics_exp001 = _bootstrap_metric_dict(sample_true, sample_pred_exp001, bootstrap_labels)
    sample_metrics_exp002 = _bootstrap_metric_dict(sample_true, sample_pred_exp002, bootstrap_labels)

    for metric_name in metric_names:
        value_exp001 = sample_metrics_exp001[metric_name]
        value_exp002 = sample_metrics_exp002[metric_name]
        bootstrap_rows.append({
            'replicate': replicate,
            'metric': metric_name,
            'exp001': value_exp001,
            'exp002': value_exp002,
            'diff_exp002_minus_exp001': value_exp002 - value_exp001,
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df.to_csv(bootstrap_samples_csv, index=False)

alpha = 1.0 - BOOTSTRAP_CI
summary_rows = []

for metric_name in metric_names:
    metric_boot = bootstrap_df[bootstrap_df['metric'] == metric_name]
    exp001_values = metric_boot['exp001'].to_numpy()
    exp002_values = metric_boot['exp002'].to_numpy()
    diff_values = metric_boot['diff_exp002_minus_exp001'].to_numpy()

    prob_exp002_le_exp001 = float(np.mean(diff_values <= 0))
    two_sided_sign_probability = float(2 * min(
        np.mean(diff_values <= 0),
        np.mean(diff_values >= 0),
    ))

    summary_rows.append({
        'Metric': metric_name,
        'Observed exp001': observed_exp001[metric_name],
        'Observed exp002': observed_exp002[metric_name],
        'Observed Δ (exp002 − exp001)': observed_exp002[metric_name] - observed_exp001[metric_name],
        'Bootstrap mean Δ': float(np.mean(diff_values)),
        'Bootstrap std Δ': float(np.std(diff_values, ddof=1)),
        'exp001 CI low': float(np.quantile(exp001_values, alpha / 2)),
        'exp001 CI high': float(np.quantile(exp001_values, 1 - alpha / 2)),
        'exp002 CI low': float(np.quantile(exp002_values, alpha / 2)),
        'exp002 CI high': float(np.quantile(exp002_values, 1 - alpha / 2)),
        'Δ CI low': float(np.quantile(diff_values, alpha / 2)),
        'Δ CI high': float(np.quantile(diff_values, 1 - alpha / 2)),
        'P(Δ <= 0)': prob_exp002_le_exp001,
        'Two-sided sign probability': min(two_sided_sign_probability, 1.0),
    })

bootstrap_summary_df = pd.DataFrame(summary_rows).set_index('Metric')
bootstrap_summary_df.to_csv(bootstrap_summary_csv)

bootstrap_payload = {
    'method': 'stratified paired bootstrap over held-out test rows',
    'n_bootstrap': N_BOOTSTRAP,
    'ci_level': BOOTSTRAP_CI,
    'random_state': BOOTSTRAP_RANDOM_STATE,
    'n_test_samples': int(len(y_true_boot)),
    'class_labels': [int(label) for label in bootstrap_labels],
    'class_counts': {
        str(int(label)): int(np.sum(y_true_boot == label))
        for label in bootstrap_labels
    },
    'summary': bootstrap_summary_df.reset_index().to_dict(orient='records'),
}

with open(bootstrap_json, 'w') as f:
    json.dump(bootstrap_payload, f, indent=2)

print('Stratified paired bootstrap: exp001 vs exp002 final models')
print(f'Bootstrap replicates: {N_BOOTSTRAP:,} | CI level: {BOOTSTRAP_CI:.0%}')
print()
print(bootstrap_summary_df.to_string(float_format='%.4f'))
print(f"\n✓ Saved bootstrap samples → {bootstrap_samples_csv}")
print(f"✓ Saved bootstrap summary → {bootstrap_summary_csv}")
print(f"✓ Saved bootstrap JSON    → {bootstrap_json}")


## 11. SHAP Beeswarm Plots (Per Class)

In [ ]:
# Generate beeswarm plots
scaler = finetuned_model.named_steps['scaler']
X_test_scaled = scaler.transform(X_test_k)

shap_values = importance_results['shap_values']
explainer = importance_results['explainer']

figs = visualization.plot_shap_beeswarm(
    shap_values, explainer, X_test_scaled, humanized_features, class_names,
    output_dir=LATEX_FIGURES_EXT,
)

print(f"\n✓ Generated {len(figs)} beeswarm plots → {LATEX_FIGURES_EXT}")

## 12. Export All PDF Figures

Generate publication-quality PDF figures from saved results.
Unlike exp001, we call individual plot functions to source data from `results_ext/`.

In [ ]:
LATEX_FIGURES_EXT.mkdir(parents=True, exist_ok=True)
export_results = {}

# ── Load data from results_ext/ ──────────────────────────────────────────────
k_summary_df = pd.read_csv(RFE_EXT / "rfe_k_path_summary.csv")
with open(RFE_EXT / "rfe_k_selection.json", 'r') as f:
    k_sel = json.load(f)

impurity_df = pd.read_csv(IMPORTANCE_EXT / "impurity_importance.csv")
perm_df     = pd.read_csv(IMPORTANCE_EXT / "permutation_importance.csv")
shap_df     = pd.read_csv(IMPORTANCE_EXT / "shap_importance.csv")
ranking_df  = pd.read_csv(IMPORTANCE_EXT / "ranking_comparison.csv")
shap_per_class_df = pd.read_csv(IMPORTANCE_EXT / "shap_importance_per_class.csv", index_col=0)

cm_df = pd.read_csv(METRICS_EXT / "confusion_matrix_final.csv", index_col=0)
cm    = cm_df.values.astype(int)
class_labels = list(cm_df.index)
k_1se = k_sel['k_1se']

# ── Generate individual figures ──────────────────────────────────────────────
try:
    fig = visualization.plot_k_path(k_summary_df, k_sel, zoomed=True, output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['k_path_cv'] = True; print("  ✓ k_path_cv.pdf")
except Exception as e:
    export_results['k_path_cv'] = False; print(f"  ✗ k_path_cv: {e}")

try:
    fig = visualization.plot_k_path(k_summary_df, k_sel, zoomed=False, output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['full_k_path_cv'] = True; print("  ✓ full_k_path_cv.pdf")
except Exception as e:
    export_results['full_k_path_cv'] = False; print(f"  ✗ full_k_path_cv: {e}")

try:
    fig = visualization.plot_importance_bar(impurity_df, 'importance',
        'XGBoost Feature Importance (Impurity-based)', 'Importance (Gain)', 'darkorange',
        filename='impurity_importance', output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['impurity_importance'] = True; print("  ✓ impurity_importance.pdf")
except Exception as e:
    export_results['impurity_importance'] = False; print(f"  ✗ impurity_importance: {e}")

try:
    imp_col = 'importance_mean' if 'importance_mean' in perm_df.columns else 'importance'
    std_col = 'importance_std' if 'importance_std' in perm_df.columns else None
    fig = visualization.plot_importance_bar(perm_df, imp_col,
        'Permutation Feature Importance', 'Importance (Mean Accuracy Decrease)', 'teal', std_col,
        filename='permutation_importance', output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['permutation_importance'] = True; print("  ✓ permutation_importance.pdf")
except Exception as e:
    export_results['permutation_importance'] = False; print(f"  ✗ permutation_importance: {e}")

try:
    fig = visualization.plot_importance_bar(shap_df, 'importance',
        'SHAP Feature Importance', 'Mean |SHAP Value|', 'forestgreen',
        filename='shap_importance_bar', output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['shap_importance_bar'] = True; print("  ✓ shap_importance_bar.pdf")
except Exception as e:
    export_results['shap_importance_bar'] = False; print(f"  ✗ shap_importance_bar: {e}")

try:
    fig = visualization.plot_ranking_comparison(ranking_df, output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['ranking_comparison'] = True; print("  ✓ ranking_comparison.pdf")
except Exception as e:
    export_results['ranking_comparison'] = False; print(f"  ✗ ranking_comparison: {e}")

try:
    fig = visualization.plot_confusion_matrix(cm, class_labels, normalized=False, k=k_1se, output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['confusion_matrix_counts'] = True; print("  ✓ confusion_matrix_counts.pdf")
except Exception as e:
    export_results['confusion_matrix_counts'] = False; print(f"  ✗ confusion_matrix_counts: {e}")

try:
    fig = visualization.plot_confusion_matrix(cm, class_labels, normalized=True, k=k_1se, output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['confusion_matrix_normalized'] = True; print("  ✓ confusion_matrix_normalized.pdf")
except Exception as e:
    export_results['confusion_matrix_normalized'] = False; print(f"  ✗ confusion_matrix_normalized: {e}")

try:
    fig = visualization.plot_shap_by_class(shap_per_class_df, output_dir=LATEX_FIGURES_EXT)
    plt.close(fig); export_results['shap_importance_by_class'] = True; print("  ✓ shap_importance_by_class.pdf")
except Exception as e:
    export_results['shap_importance_by_class'] = False; print(f"  ✗ shap_importance_by_class: {e}")

n_ok = sum(export_results.values())
print(f"\n✓ Exported {n_ok}/{len(export_results)} figures to {LATEX_FIGURES_EXT}")

## 13. Export LaTeX Tables

In [ ]:
# Prepare grid results for LaTeX export
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)
class_names = sorted(test_ext_df['category'].unique())

# Load selected features
with open(FINETUNE_EXT / f"selected_features_k{SELECTED_K}.txt", 'r') as f:
    features_k = [line.strip() for line in f.readlines()]

# Evaluate model
model = joblib.load(FINETUNE_EXT / f"xgb_finetuned_k{SELECTED_K}.pkl")
X_test_k = test_ext_df[features_k].values
y_test   = test_ext_df['target'].values
y_pred   = model.predict(X_test_k)

metrics, cm, per_class = modeling.compute_all_metrics(y_test, y_pred, class_names)

grid_results = {
    f'XGB_k{SELECTED_K}_finetuned_ext': {
        'metrics': metrics,
        'cm': cm,
        'per_class': per_class,
        'class_names': class_names,
    }
}

# Export → results_ext/latex_tables/
exported = visualization.export_latex_tables(grid_results, output_dir=LATEX_TABLES_EXT)
print(f"\n✓ Exported {len(exported)} LaTeX files to: {LATEX_TABLES_EXT}")

## 14. Export XGBoost to C Code

In [ ]:
# Export fine-tuned XGBoost to C code → results_ext/tl2cgen_export/
model = joblib.load(FINETUNE_EXT / f"xgb_finetuned_k{SELECTED_K}.pkl")

result = visualization.export_xgb_to_c(model, f"xgb_k{SELECTED_K}_finetuned_ext", TL2CGEN_EXT)

if result:
    size, lines = result
    print(f"\n✓ C code exported to: {TL2CGEN_EXT}")
    print(f"  Size: {size/1024:.2f} KB, Lines: {lines:,}")
else:
    print("✗ tl2cgen export failed (check if treelite/tl2cgen is installed)")

## 15. Soft-Voting Ensemble (XGB + RF + SVM)

Soft-voting ensemble combining the three best single classifiers using the
RFE-selected feature set from the extended pipeline.
Best hyperparameters are loaded from `results_ext/grid_search/`.

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)

with open(FINETUNE_EXT / f"selected_features_k{SELECTED_K}.txt") as f:
    features_sel = [l.strip() for l in f if l.strip()]

X_train_k = train_ext_df[features_sel].values
y_train   = train_ext_df["target"].values
X_test_k  = test_ext_df[features_sel].values
y_test    = test_ext_df["target"].values

all_classes = np.unique(y_train)
class_mapping = train_ext_df[["category", "target"]].drop_duplicates().set_index("target")["category"].to_dict()
class_names = [class_mapping[int(c)] for c in all_classes]

# ── Load best params from results_ext/grid_search/ ───────────────────────────
with open(FINETUNE_EXT / f"xgb_finetuned_k{SELECTED_K}_results.json") as f:
    xgb_stats = json.load(f)
xgb_params = {k.replace("clf__", ""): v for k, v in xgb_stats.get("best_params", {}).items()}

def load_best_params_grid(model_name: str) -> dict:
    path = GRID_SEARCH_EXT / f"{model_name}_fold_statistics.json"
    with open(path) as f:
        stats = json.load(f)
    return {k.replace("clf__", ""): v for k, v in stats["best_params"].items()}

rf_params  = load_best_params_grid("RF")
svm_params = load_best_params_grid("SVM")
svm_params["probability"] = True

skf = StratifiedKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)

per_fold_metrics = []
probas_aligned = []

print(f"Running {config.CV_FOLDS}-fold CV training and evaluating on test set (k={len(features_sel)})...")

for fold_idx, (train_idx, _) in enumerate(skf.split(X_train_k, y_train)):
    print(f"\nFold {fold_idx+1}/{config.CV_FOLDS}")
    X_tr = X_train_k[train_idx]
    y_tr = y_train[train_idx]

    xgb_est = XGBClassifier(**xgb_params, random_state=config.RANDOM_STATE, eval_metric="logloss")
    rf_est  = RandomForestClassifier(**rf_params, random_state=config.RANDOM_STATE)
    svm_est = SVC(**svm_params, random_state=config.RANDOM_STATE)

    voting_clf = VotingClassifier(
        estimators=[("xgb", xgb_est), ("rf", rf_est), ("svm", svm_est)],
        voting="soft", n_jobs=-1,
    )
    pipeline = Pipeline([("scaler", StandardScaler()), ("clf", voting_clf)])
    pipeline.fit(X_tr, y_tr)

    clf = pipeline.named_steps["clf"]
    proba = clf.predict_proba(pipeline.named_steps["scaler"].transform(X_test_k))
    fold_classes = clf.classes_

    n_test = X_test_k.shape[0]
    n_classes = len(all_classes)
    aligned = np.zeros((n_test, n_classes), dtype=float)
    for col_idx, cls in enumerate(fold_classes):
        target_col = int(np.where(all_classes == cls)[0][0])
        aligned[:, target_col] = proba[:, col_idx]

    probas_aligned.append(aligned)

    y_pred_fold = np.argmax(aligned, axis=1)
    m, cm_fold, per_class = modeling.compute_all_metrics(y_test, y_pred_fold, class_names)
    per_fold_metrics.append({"fold": fold_idx + 1, **m})

    print(f"  Fold Test F1 (Macro): {m['F1 (Macro)']:.4f}")

# ── Aggregate across folds ────────────────────────────────────────────────────
probas_mean = np.mean(np.stack(probas_aligned, axis=0), axis=0)
y_pred_mean = np.argmax(probas_mean, axis=1)

metrics_mean, cm_mean, per_class_mean = modeling.compute_all_metrics(y_test, y_pred_mean, class_names)

print(f"\nAggregated Test Results (averaged probabilities):")
print(f"  Accuracy      : {metrics_mean['Accuracy']:.4f}")
print(f"  Balanced Acc. : {metrics_mean['Balanced Accuracy']:.4f}")
print(f"  F1 (Macro)    : {metrics_mean['F1 (Macro)']:.4f}")

# Compare with fine-tuned XGB baseline
with open(FINETUNE_EXT / f"xgb_finetuned_k{SELECTED_K}_results.json") as f:
    xgb_baseline = json.load(f)
xgb_f1 = xgb_baseline.get("test_score", None)
if xgb_f1 is not None:
    delta = metrics_mean["F1 (Macro)"] - xgb_f1
    print(f"Fine-tuned XGB k={SELECTED_K} F1 (Macro): {xgb_f1:.4f}")
    print(f"Ensemble F1 (Macro)             : {metrics_mean['F1 (Macro)']:.4f}  (Δ {delta:+.4f})")

# ── Save ──────────────────────────────────────────────────────────────────────
ENSEMBLE_EXT.mkdir(parents=True, exist_ok=True)

per_fold_df = pd.DataFrame(per_fold_metrics)
per_fold_df.to_csv(ENSEMBLE_EXT / "ensemble_per_fold_metrics.csv", index=False)

agg_df = pd.DataFrame({"Metric": list(metrics_mean.keys()), "Value": list(metrics_mean.values())})
agg_df.to_csv(ENSEMBLE_EXT / "ensemble_aggregated_metrics.csv", index=False)
agg_df.to_csv(ENSEMBLE_EXT / "ensemble_metrics.csv", index=False)

cm_df = pd.DataFrame(cm_mean, index=class_names, columns=class_names)
cm_df.index.name = "True"
cm_df.to_csv(ENSEMBLE_EXT / "ensemble_confusion_matrix.csv")

print(f"\n✓ Saved ensemble results to: {ENSEMBLE_EXT}")

### 15b. Experiment Comparison — exp001 vs exp002 Ensemble

Side-by-side metrics comparing the exp001 soft-voting ensemble (k=17 FRESH features)
with the exp002 ensemble (k=RFE-selected, FRESH + extended features).


In [ ]:
# ── Load exp001 ensemble aggregated metrics ───────────────────────────────────
_e1_agg_path = config.ENSEMBLE_DIR / "ensemble_aggregated_metrics.csv"
if _e1_agg_path.exists():
    _e1_df = pd.read_csv(_e1_agg_path).set_index("Metric")["Value"]
else:
    _e1_df = None
    print(f"⚠ exp001 ensemble metrics not found: {_e1_agg_path}")

# ── Load exp001 per-fold metrics ──────────────────────────────────────────────
_e1_fold_path = config.ENSEMBLE_DIR / "ensemble_per_fold_metrics.csv"
if _e1_fold_path.exists():
    _e1_fold_df = pd.read_csv(_e1_fold_path)
else:
    _e1_fold_df = None

# ── Load exp002 ensemble aggregated metrics (just saved) ─────────────────────
_e2_agg_path = ENSEMBLE_EXT / "ensemble_aggregated_metrics.csv"
if _e2_agg_path.exists():
    _e2_df = pd.read_csv(_e2_agg_path).set_index("Metric")["Value"]
else:
    _e2_df = pd.Series(metrics_mean)

# ── Load exp002 per-fold metrics ──────────────────────────────────────────────
_e2_fold_path = ENSEMBLE_EXT / "ensemble_per_fold_metrics.csv"
if _e2_fold_path.exists():
    _e2_fold_df = pd.read_csv(_e2_fold_path)
else:
    _e2_fold_df = pd.DataFrame(per_fold_metrics)

# ── Build comparison table ────────────────────────────────────────────────────
_metric_keys = ["Accuracy", "Balanced Accuracy", "F1 (Macro)", "F1 (Weighted)",
                "Precision (Macro)", "Recall (Macro)"]

_rows = []
for _mk in _metric_keys:
    _v1 = float(_e1_df.get(_mk, float("nan"))) if _e1_df is not None else float("nan")
    _v2 = float(_e2_df.get(_mk, float("nan")))
    _delta = _v2 - _v1
    _rows.append({"Metric": _mk,
                  "exp001 (k=17)":      round(_v1,    4),
                  f"exp002 (k={SELECTED_K}, ext)": round(_v2,    4),
                  "Δ (exp002 − exp001)": round(_delta, 4)})

comparison_df = pd.DataFrame(_rows).set_index("Metric")

# ── Per-fold std rows ─────────────────────────────────────────────────────────
_pf_rows = []
for _mk in ["Accuracy", "F1 (Macro)"]:
    _std1 = float(_e1_fold_df[_mk].std()) if _e1_fold_df is not None else float("nan")
    _std2 = float(_e2_fold_df[_mk].std()) if _e2_fold_df is not None else float("nan")
    _pf_rows.append({"Metric": f"{_mk} (std across folds)",
                     "exp001 (k=17)":      round(_std1, 4),
                     f"exp002 (k={SELECTED_K}, ext)": round(_std2, 4),
                     "Δ (exp002 − exp001)": round(_std2 - _std1, 4)})

comparison_std_df = pd.DataFrame(_pf_rows).set_index("Metric")

# ── Print & save ──────────────────────────────────────────────────────────────
print("="*70)
print("  Ensemble Comparison: exp001 (k=17)  vs  exp002 (k=%d, ext)" % SELECTED_K)
print("="*70)
print(comparison_df.to_string(float_format="%.4f"))
print()
print(comparison_std_df.to_string(float_format="%.4f"))
print()

full_comparison_df = pd.concat([comparison_df, comparison_std_df])
full_comparison_df.index.name = "Metric"
full_comparison_df.to_csv(ENSEMBLE_EXT / "ensemble_comparison_exp001_vs_exp002.csv")
print(f"✓ Saved comparison table → {ENSEMBLE_EXT / 'ensemble_comparison_exp001_vs_exp002.csv'}")


## 16. Conformal Prediction (MAPIE — CrossConformalClassifier)

Uncertainty quantification using cross-conformal prediction on the fine-tuned
XGBoost model from the extended pipeline.

- **CrossConformalClassifier** uses 5-fold CV internally
- **Prediction sets** replace single-label outputs
- Evaluated at three confidence levels (α = 0.10, 0.05, 0.01) with both **LAC** and **APS**

In [ ]:
# ── Load data & features ──────────────────────────────────────────────────────
train_ext_df = pd.read_csv(EXTENDED_TRAIN_CSV)
test_ext_df  = pd.read_csv(EXTENDED_TEST_CSV)

with open(FINETUNE_EXT / f"selected_features_k{SELECTED_K}.txt") as f:
    features_sel = [l.strip() for l in f if l.strip()]

X_train_k = train_ext_df[features_sel].values
y_train   = train_ext_df["target"].values
X_test_k  = test_ext_df[features_sel].values
y_test    = test_ext_df["target"].values
n_test    = len(y_test)

class_mapping = train_ext_df[["category", "target"]].drop_duplicates().set_index("target")["category"].to_dict()

# ── Load fine-tuned XGB params ────────────────────────────────────────────────
with open(FINETUNE_EXT / f"xgb_finetuned_k{SELECTED_K}_results.json") as f:
    xgb_stats = json.load(f)
conformal_xgb_params = {k.replace("clf__", ""): v for k, v in xgb_stats.get("best_params", {}).items()}

def make_xgb_pipeline() -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(**conformal_xgb_params, random_state=config.RANDOM_STATE, eval_metric="logloss")),
    ])

CONFIDENCE_LEVELS = [0.90, 0.95, 0.99]
SCORE_METHODS     = {"LAC": "lac", "APS": "aps"}

all_rows      = []
detailed_rows = []

CONFORMAL_EXT.mkdir(parents=True, exist_ok=True)

for score_name, score_str in SCORE_METHODS.items():
    print(f"\n{'='*55}\n  Conformity score: {score_name}\n{'='*55}")

    for conf_level in CONFIDENCE_LEVELS:
        alpha = 1.0 - conf_level

        mapie = CrossConformalClassifier(
            estimator=make_xgb_pipeline(),
            confidence_level=conf_level,
            conformity_score=score_str,
            cv=config.CV_FOLDS,
            random_state=config.RANDOM_STATE,
        )
        mapie.fit_conformalize(X_train_k, y_train)

        y_pred_mapie, y_pred_set = mapie.predict_set(X_test_k)
        set_mask  = y_pred_set[:, :, 0]
        set_sizes = set_mask.sum(axis=1)

        classes_ = getattr(mapie, "classes_", None)
        if classes_ is None:
            classes_ = getattr(mapie, "_mapie_classifier").classes_
        classes_ = np.asarray(classes_)

        class_to_col = {c: i for i, c in enumerate(classes_)}
        y_col = np.array([class_to_col[c] for c in y_test])
        cls_names = [class_mapping.get(int(c), str(c)) for c in classes_]

        coverage      = float(np.mean(set_mask[np.arange(n_test), y_col]))
        mean_set_size = float(set_sizes.mean())
        n_empty       = int((set_sizes == 0).sum())
        n_single      = int((set_sizes == 1).sum())
        n_multi       = int((set_sizes >  1).sum())

        confident_mask  = (set_sizes == 1)
        abstention_rate = 1.0 - (n_single / n_test)
        y_pred_from_set = classes_[set_mask.argmax(axis=1)]

        if n_single > 0:
            selective_acc = float(accuracy_score(
                y_test[confident_mask], y_pred_from_set[confident_mask]))
            selective_macro_f1 = float(f1_score(
                y_test[confident_mask], y_pred_from_set[confident_mask],
                labels=classes_, average="macro", zero_division=0))

            cm_selective = confusion_matrix(
                y_test[confident_mask], y_pred_from_set[confident_mask], labels=classes_)

            report = classification_report(
                y_test[confident_mask], y_pred_from_set[confident_mask],
                labels=classes_, target_names=cls_names,
                output_dict=True, zero_division=0)

            print(f"\n  α = {alpha:.2f}  (confidence = {conf_level:.0%})")
            print(classification_report(
                y_test[confident_mask], y_pred_from_set[confident_mask],
                labels=classes_, target_names=cls_names, zero_division=0))

            for cn in cls_names:
                if cn in report:
                    detailed_rows.append({
                        "method": score_name, "confidence_level": conf_level,
                        "alpha": round(alpha, 2), "class": cn,
                        "precision": round(report[cn]["precision"], 4),
                        "recall":    round(report[cn]["recall"], 4),
                        "f1_score":  round(report[cn]["f1-score"], 4),
                        "support":   int(report[cn]["support"]),
                    })
        else:
            selective_acc = selective_macro_f1 = float("nan")
            cm_selective = None

        ABSTAIN = -1
        y_pred_with_abstain = y_pred_from_set.copy()
        y_pred_with_abstain[~confident_mask] = ABSTAIN
        labels_with_abstain = np.append(classes_, ABSTAIN)
        cm_with_abstain = confusion_matrix(y_test, y_pred_with_abstain, labels=labels_with_abstain)

        print(f"  Coverage: {coverage:.4f} | Abstention: {abstention_rate:.4f} | "
              f"Selective Acc: {selective_acc:.4f} | Selective Macro-F1: {selective_macro_f1:.4f}")

        tag = f"{score_name.lower()}_{int(conf_level * 100)}"
        if cm_selective is not None:
            pd.DataFrame(cm_selective, index=cls_names, columns=cls_names).to_csv(
                CONFORMAL_EXT / f"cm_selective_{tag}.csv", index_label="True")

        cls_names_ext = cls_names + ["Abstain"]
        pd.DataFrame(cm_with_abstain, index=cls_names_ext, columns=cls_names_ext).to_csv(
            CONFORMAL_EXT / f"cm_with_abstain_{tag}.csv", index_label="True")

        all_rows.append({
            "method": score_name, "confidence_level": conf_level,
            "alpha": round(alpha, 4), "coverage": round(coverage, 4),
            "mean_set_size": round(mean_set_size, 4),
            "n_empty": n_empty, "n_singleton": n_single, "n_multi": n_multi,
            "abstention_rate": round(abstention_rate, 4),
            "selective_accuracy": round(selective_acc, 4) if not np.isnan(selective_acc) else None,
            "selective_macro_f1": round(selective_macro_f1, 4) if not np.isnan(selective_macro_f1) else None,
        })

results_df = pd.DataFrame(all_rows)
results_df.to_csv(CONFORMAL_EXT / "conformal_results.csv", index=False)

detailed_df = pd.DataFrame(detailed_rows)
detailed_df.to_csv(CONFORMAL_EXT / "conformal_selective_per_class.csv", index=False)

print(f"\n{'='*55}")
print(f"✓ Saved conformal summary    ({len(results_df)} rows)  → {CONFORMAL_EXT}")
print(f"✓ Saved per-class detail     ({len(detailed_df)} rows) → {CONFORMAL_EXT}")

## 17. Export LaTeX Tables — Ensemble & Conformal

Generate LaTeX tables for the soft-voting ensemble and conformal prediction results.

**Ensemble tables**: Per-fold metrics, confusion matrix, per-class metrics.
**Conformal tables**: Summary table and per-class selective metrics.

In [ ]:
output_dir = LATEX_TABLES_EXT
output_dir.mkdir(parents=True, exist_ok=True)
exported = []

# ══════════════════════════════════════════════════════════════════════════════
# A. ENSEMBLE TABLES
# ══════════════════════════════════════════════════════════════════════════════

# ── A1. Per-fold metrics table ────────────────────────────────────────────────
fold_df = pd.read_csv(ENSEMBLE_EXT / "ensemble_per_fold_metrics.csv")

fold_table_cols = ["fold", "Accuracy", "F1 (Macro)", "F1 (Weighted)",
                   "Precision (Macro)", "Recall (Macro)"]
fold_table = fold_df[fold_table_cols].copy()
fold_table["fold"] = fold_table["fold"].astype(int).astype(str)

mean_row = fold_table.drop(columns="fold").mean()
std_row  = fold_table.drop(columns="fold").std()
fold_table = pd.concat([
    fold_table,
    pd.DataFrame([["Mean"] + mean_row.tolist()], columns=fold_table.columns),
    pd.DataFrame([["Std"]  + std_row.tolist()],  columns=fold_table.columns),
], ignore_index=True)
fold_table.rename(columns={"fold": "Fold"}, inplace=True)

fold_tex = fold_table.to_latex(
    index=False, float_format="%.4f",
    caption="Soft-Voting Ensemble (Extended Features): Per-fold and Summary Metrics",
    label="tab:ensemble_ext_per_fold",
)
out_path = output_dir / "ensemble_per_fold_metrics.tex"
out_path.write_text(fold_tex)
exported.append(out_path.name)

# ── A2. Ensemble confusion matrix ────────────────────────────────────────────
cm_df = pd.read_csv(ENSEMBLE_EXT / "ensemble_confusion_matrix.csv", index_col=0)
cm_df.columns.name = "Predicted"
cm_df.index.name   = "True"

cm_tex = cm_df.to_latex(
    caption=f"Confusion Matrix - Ensemble (XGB+RF+SVM, k={SELECTED_K}, Extended)",
    label="tab:ensemble_ext_cm",
)
out_path = output_dir / f"confusion_matrix_ensemble_ext_k{SELECTED_K}.tex"
out_path.write_text(cm_tex)
exported.append(out_path.name)

# ── A3. Ensemble per-class metrics ───────────────────────────────────────────
ensemble_metrics_df = pd.read_csv(ENSEMBLE_EXT / "ensemble_metrics.csv")

cm = cm_df.values.astype(int)
class_names_cm = list(cm_df.index)

y_true_recon, y_pred_recon = [], []
for i, true_cls in enumerate(class_names_cm):
    for j, pred_cls in enumerate(class_names_cm):
        y_true_recon.extend([true_cls] * cm[i, j])
        y_pred_recon.extend([pred_cls] * cm[i, j])

per_class_report = classification_report(
    y_true_recon, y_pred_recon,
    labels=class_names_cm, output_dict=True, zero_division=0,
)

pc_rows = []
for cn in class_names_cm:
    pc_rows.append({
        "": cn,
        "precision": per_class_report[cn]["precision"],
        "recall":    per_class_report[cn]["recall"],
        "f1-score":  per_class_report[cn]["f1-score"],
        "support":   per_class_report[cn]["support"],
    })
for avg_key in ["accuracy", "macro avg", "weighted avg"]:
    if avg_key in per_class_report:
        entry = per_class_report[avg_key]
        if isinstance(entry, dict):
            pc_rows.append({"": avg_key, **{k: entry[k] for k in ["precision", "recall", "f1-score", "support"]}})
        else:
            pc_rows.append({"": avg_key, "precision": entry, "recall": entry, "f1-score": entry, "support": entry})

pc_df = pd.DataFrame(pc_rows).set_index("")
pc_tex = pc_df.to_latex(
    float_format="%.4f",
    caption=f"Per-class Metrics - Ensemble (XGB+RF+SVM, k={SELECTED_K}, Extended)",
    label="tab:ensemble_ext_per_class",
)
out_path = output_dir / f"per_class_ensemble_ext_k{SELECTED_K}.tex"
out_path.write_text(pc_tex)
exported.append(out_path.name)

# ══════════════════════════════════════════════════════════════════════════════
# B. CONFORMAL TABLES
# ══════════════════════════════════════════════════════════════════════════════

# ── B1. Conformal summary ────────────────────────────────────────────────────
conformal_df = pd.read_csv(CONFORMAL_EXT / "conformal_results.csv")

col_rename = {
    "method": "Score", "confidence_level": "Confidence", "alpha": r"$\alpha$",
    "coverage": "Coverage", "mean_set_size": "Mean Set Size",
    "n_empty": "Empty", "n_singleton": "Singleton", "n_multi": "Multi-label",
    "abstention_rate": "Abstention",
    "selective_accuracy": "Selective Acc.", "selective_macro_f1": "Selective F1",
}
conf_pub = conformal_df.rename(columns=col_rename)
conf_pub["Confidence"] = conf_pub["Confidence"].apply(lambda x: f"{x:.0%}")

cols = [c for c in col_rename.values() if c in conf_pub.columns]
conf_pub = conf_pub[cols]

conf_tex = conf_pub.to_latex(
    index=False, float_format="%.4f",
    caption=(
        f"Conformal prediction results (CrossConformalClassifier, XGBoost k={SELECTED_K}, Extended). "
        "Coverage is the empirical marginal coverage on the test set."
    ),
    label="tab:conformal_ext_results", escape=False,
)
out_path = output_dir / "conformal_results_ext.tex"
out_path.write_text(conf_tex)
exported.append(out_path.name)

# ── B2. Conformal per-class selective metrics ────────────────────────────────
detailed_path = CONFORMAL_EXT / "conformal_selective_per_class.csv"
if detailed_path.exists():
    detailed_df = pd.read_csv(detailed_path)
    if not detailed_df.empty:
        col_rename2 = {
            "method": "Score", "confidence_level": "Confidence", "alpha": r"$\alpha$",
            "class": "Class", "precision": "Precision", "recall": "Recall",
            "f1_score": "F1-score", "support": "Support",
        }
        detailed_pub = detailed_df.rename(columns=col_rename2)
        detailed_pub["Confidence"] = detailed_pub["Confidence"].apply(lambda x: f"{x:.0%}")
        cols = [c for c in col_rename2.values() if c in detailed_pub.columns]
        detailed_pub = detailed_pub[cols]

        det_tex = detailed_pub.to_latex(
            index=False, float_format="%.4f",
            caption="Per-class selective metrics under conformal prediction (Extended Features).",
            label="tab:conformal_ext_per_class", escape=False,
        )
        out_path = output_dir / "conformal_selective_per_class_ext.tex"
        out_path.write_text(det_tex)
        exported.append(out_path.name)

print(f"\n✓ Exported {len(exported)} LaTeX tables to: {output_dir}")
for name in exported:
    print(f"  • {name}")

## 18. Exploration: k=17 RFE Features + Extended Audio Features

As a supplementary experiment, we combine the **k=17 RFE-selected features** from exp001
(the smallest high-performing subset) with the **extended audio features** computed in
Section 1. This 35-feature set tests whether the most informative TSFresh descriptors,
when augmented with the new audio features, can match the full FRESH+ext pipeline while
remaining more parsimonious.

We reuse the best XGBoost hyperparameters found for the main pipeline and report a
side-by-side comparison of three configurations:
1. **k=17 only** — baseline from exp001 (loaded from saved metrics)
2. **k=17 + extended** — this exploration
3. **Full FRESH + extended** — main exp002 pipeline (loaded from saved metrics)


In [ ]:
import json, pathlib

# ── 1. Load k=17 feature names ────────────────────────────────────────────────
with open(config.FINETUNE_DIR / "selected_features_k17.txt") as fh:
    features_k17 = [l.strip() for l in fh if l.strip()]
print(f"k=17 features: {len(features_k17)}")

# ── 2. Reload FRESH train / test splits (with all features) ──────────────────
fresh_train_df = pd.read_csv(config.FRESH_TRAIN)
fresh_test_df  = pd.read_csv(config.FRESH_TEST)

# ── 3. Recover sample filenames (same deterministic split) ───────────────────
_dc  = pd.read_csv(config.DATASET_CLEAN).copy()
_con = pd.read_csv(config.CONSOLIDATED_SUMMARY)
_dc['filename'] = _con['filename'].values[:len(_dc)]

_y_all = _dc['target']
_idx   = np.arange(len(_dc))
_tr_idx, _te_idx = train_test_split(
    _idx, test_size=config.TEST_SIZE, stratify=_y_all,
    random_state=config.RANDOM_STATE
)
_tr_stems = _dc.iloc[_tr_idx]['filename'].str.replace('.wav', '').values
_te_stems  = _dc.iloc[_te_idx]['filename'].str.replace('.wav', '').values

# ── 4. Align extended features ───────────────────────────────────────────────
_ext_cols = [c for c in extended_df.columns if c not in ['filename', 'category', 'target']]
_ext_lut  = (
    extended_df
    .assign(stem=extended_df['filename'].str.replace('.wav', ''))
    .set_index('stem')[_ext_cols]
)
_ext_tr = _ext_lut.reindex(_tr_stems).fillna(0).reset_index(drop=True)
_ext_te = _ext_lut.reindex(_te_stems).fillna(0).reset_index(drop=True)

# ── 5. Build k17+ext DataFrames ──────────────────────────────────────────────
k17_combined = features_k17 + _ext_cols

train_k17ext = pd.concat(
    [fresh_train_df[['category'] + features_k17 + ['target']].reset_index(drop=True),
     _ext_tr],
    axis=1,
)
test_k17ext = pd.concat(
    [fresh_test_df[['category'] + features_k17 + ['target']].reset_index(drop=True),
     _ext_te],
    axis=1,
)
print(f"k17+ext combined: {len(k17_combined)} features  |  train {len(train_k17ext)}  test {len(test_k17ext)}")

# ── 6. Load best XGB hyperparameters from the main exp002 grid search ─────────
_gs_dir   = GRID_SEARCH_EXT
_gs_files = sorted(_gs_dir.glob("xgb_gs_k*.json"))
_best_params, _best_k = None, None
for _f in _gs_files:
    _res = json.loads(_f.read_text())
    if _best_params is None or _res['best_score'] > _best_params['best_score']:
        _best_params = _res
        _best_k = int(_f.stem.split('_k')[-1])

if _best_params is None:
    print("⚠ No grid-search results found in GRID_SEARCH_EXT — using defaults.")
    _xgb_params = {}
else:
    _xgb_params = _best_params['best_params']
    print(f"Best XGB params from exp002 grid search (k={_best_k}): {_xgb_params}")

# ── 7. Fit + evaluate XGBoost on k17+ext ─────────────────────────────────────
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

_le   = LabelEncoder()
_y_tr = _le.fit_transform(train_k17ext['target'])
_y_te = _le.transform(test_k17ext['target'])

_X_tr = train_k17ext[k17_combined].values
_X_te = test_k17ext[k17_combined].values

_clf = XGBClassifier(
    **_xgb_params,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=config.RANDOM_STATE,
    n_jobs=-1,
)
_clf.fit(_X_tr, _y_tr)

_y_pred = _clf.predict(_X_te)
_f1_k17ext = f1_score(_y_te, _y_pred, average='macro')
print(f"\nk=17 + extended  →  macro-F1 = {_f1_k17ext:.4f}")

# ── 8. Load reference metrics for comparison ─────────────────────────────────
def _load_f1(path):
    try:
        d = json.loads(pathlib.Path(path).read_text())
        return d.get('macro_f1') or d.get('test_macro_f1') or d.get('f1_macro')
    except Exception:
        return None

_f1_k17 = _load_f1(config.FINETUNE_DIR / "xgb_finetuned_k17_results.json")

_ft_ext_dir = FINETUNE_EXT
_ft_ext_files = sorted(_ft_ext_dir.glob("xgb_finetuned_k*.json"))
_f1_fresh_ext = None
if _ft_ext_files:
    _f1_fresh_ext = _load_f1(_ft_ext_files[-1])

# ── 9. Print comparison table ────────────────────────────────────────────────
print("\n" + "="*55)
print(f"{'Configuration':<32}  {'Macro-F1':>8}")
print("-"*55)
if _f1_k17 is not None:
    print(f"{'k=17 only  (exp001 baseline)':<32}  {_f1_k17:>8.4f}")
else:
    print(f"{'k=17 only  (exp001 baseline)':<32}  {'N/A':>8}")
print(f"{'k=17 + extended  (this section)':<32}  {_f1_k17ext:>8.4f}")
if _f1_fresh_ext is not None:
    print(f"{'Full FRESH + extended  (exp002)':<32}  {_f1_fresh_ext:>8.4f}")
else:
    print(f"{'Full FRESH + extended  (exp002)':<32}  {'N/A (run pipeline first)':>8}")
print("="*55)
